# WLS回归可视化

用于检查基金久期测算中WLS回归的拟合效果，诊断测算久期与Wind披露久期偏差的原因。

## 功能说明
1. **第一页**：回归结果概览（2x2布局）
   - 时间序列图：基金收益率 vs 拟合值
   - 残差分布直方图
   - 因子系数条形图（含久期标注）
   - 残差时间序列图

2. **第二页**：因子散点图
   - 每个因子单独一个散点图
   - 因子收益率 vs 基金收益率
   - 点的大小表示权重

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from data_preprocessing import FundDataPreprocessor
from fund_type_classifier import FundTypeClassifier
from wind_data_fetcher import WindDataFetcher
from bond_index_data import BondIndexDataProcessor
from duration_model import FundDurationCalculator
from data_cache import WindDataCache
from wls_visualizer import WLSVisualizer

print('模块导入成功')

模块导入成功


## 1. 初始化各模块

In [4]:
# 先设置默认日期（会在cell-5中被覆盖）
if 'TARGET_DATE' not in globals():
    TARGET_DATE = '2025-12-31'

# 1. 数据预处理器
preprocessor = FundDataPreprocessor(
    short_term_path='短期纯债基金样本数据.xlsx',
    medium_long_term_path='中长期纯债基金样本数据.xlsx'
)

# 2. 基金类型分类器
classifier = FundTypeClassifier(
    holdings_path='纯债基金持仓情况.xlsx'
)
classifier.load_holdings_data()

# 3. Wind数据获取器（带本地缓存）
cache = WindDataCache(cache_dir='data')
wind_fetcher = WindDataFetcher(cache=cache)

# 4. 中债指数数据处理器
index_processor = BondIndexDataProcessor(
    index_path='中债财富指数.xlsx'
)
index_processor.load_price_data()
index_processor.load_duration_data()

# 5. 创建久期计算器
calculator = FundDurationCalculator(
    data_preprocessor=preprocessor,
    fund_classifier=classifier,
    wind_fetcher=wind_fetcher,
    index_processor=index_processor
)

# 6. 配置日志文件路径
log_file = f'./output/久期迭代日志_{TARGET_DATE.replace("-", "")}.xlsx'

# 7. 创建可视化器（传入日志文件）
visualizer = WLSVisualizer(calculator.duration_model, calculator.index_processor, log_file=log_file)

# 8. 配置输出目录
import os
output_dir = './output/wls_visualization'
os.makedirs(output_dir, exist_ok=True)

print('所有模块初始化完成')
print(f'输出目录: {output_dir}')
print(f'目标日期: {TARGET_DATE}')
if os.path.exists(log_file):
    print(f'日志文件: {log_file}')
else:
    print(f'警告: 未找到日志文件 {log_file}')

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.
Wind连接成功
从Excel加载日志: 1563 只基金
所有模块初始化完成
输出目录: ./output/wls_visualization
目标日期: 2025-12-31
日志文件: ./output/久期迭代日志_20251231.xlsx


## 2. 配置分析参数

In [5]:
# ============================================================
# 配置：修改此处以切换分析日期和基金
# ============================================================
TARGET_DATE = '2025-12-31'

# 选择要分析的基金（可以是一个或多个）
# 方式1：指定基金代码列表
SELECTED_FUNDS = [
    '000033.OF',  # 示例基金1
    '000084.OF',  # 示例基金2
    '003742.OF',  # 示例基金3（迭代轮数最多）
]

# 方式2：分析所有基金（设为True则忽略SELECTED_FUNDS，分析所有有迭代日志的基金）
ANALYZE_ALL_FUNDS = True  # 设为True分析所有基金

# 权重方法
WEIGHT_METHOD = 'linear'  # 可选：'uniform', 'linear', 'exponential'

# 是否显示图表（False则只保存不显示）
SHOW_PLOTS = False  # 批量处理时建议设为False

print(f'目标日期: {TARGET_DATE}')
if ANALYZE_ALL_FUNDS:
    print(f'模式: 分析所有基金')
else:
    print(f'待分析基金数: {len(SELECTED_FUNDS)}')

目标日期: 2025-12-31
模式: 分析所有基金


In [6]:
# 如果选择分析所有基金，从迭代日志中获取基金列表
if ANALYZE_ALL_FUNDS:
    log_file = f'./output/久期迭代日志_{TARGET_DATE.replace("-", "")}.xlsx'
    
    if os.path.exists(log_file):
        df_all = pd.read_excel(log_file, sheet_name='全量汇总')
        SELECTED_FUNDS = df_all['fund_code'].tolist()
        print(f'从迭代日志加载基金列表: {len(SELECTED_FUNDS)} 只基金')
    else:
        print(f'警告: 未找到迭代日志文件 {log_file}')
        print('将使用基金池中的所有基金')
        # 使用基金池
        fund_pool_temp = calculator.data_preprocessor.get_fund_pool(TARGET_DATE)
        SELECTED_FUNDS = []
        for fund_type in ['short', 'medium_long']:
            df = fund_pool_temp.get(fund_type, pd.DataFrame())
            SELECTED_FUNDS.extend(df['Code'].tolist())
        print(f'从基金池加载: {len(SELECTED_FUNDS)} 只基金')

print(f'最终待分析基金数: {len(SELECTED_FUNDS)}')

从迭代日志加载基金列表: 1563 只基金
最终待分析基金数: 1563


## 3. （可选）从文件加载需要分析的基金

In [7]:
# 如果需要从结果文件中选取偏差最大的基金
USE_FROM_FILE = False

if USE_FROM_FILE:
    try:
        # 读取详细结果
        results_file = f'./output/久期详细结果_linear_{TARGET_DATE.replace("-", "")}.xlsx'
        results_df = pd.read_excel(results_file)

        # 读取验证结果（如果有）
        validation_file = f'./output/验证报告.txt'
        if os.path.exists(validation_file):
            with open(validation_file, 'r', encoding='utf-8') as f:
                content = f.read()

        # 计算偏差
        results_df['bias'] = results_df['duration'] - results_df.get('reported_duration', results_df['duration'])

        # 选择偏差最大的N只基金
        TOP_N = 5
        top_bias_funds = results_df.nlargest(TOP_N, 'bias')['fund_code'].tolist()

        print(f'从文件中选取偏差最大的{TOP_N}只基金:')
        print(top_bias_funds)

        SELECTED_FUNDS = top_bias_funds
    except Exception as e:
        print(f'从文件读取失败: {e}')
        print('使用预设的基金列表')

## 4. 确保迭代日志存在

可视化需要 `iteration_logs` 数据，请确保已运行 `纯债基金久期测算.ipynb` 生成了迭代日志。

如果没有，可以先运行下面的代码加载已有日志：

In [8]:
# 尝试加载迭代日志
log_file = f'./output/久期迭代日志_{TARGET_DATE.replace("-", "")}.xlsx'

if os.path.exists(log_file):
    print(f'找到迭代日志文件: {log_file}')
    df_all = pd.read_excel(log_file, sheet_name='全量汇总')
    df_trig = pd.read_excel(log_file, sheet_name='触发汇总')
    df_detail = pd.read_excel(log_file, sheet_name='迭代详情')
    print(f'全量基金: {len(df_all)} 只')
    print(f'触发边界: {len(df_trig)} 只')
    print(f'迭代记录: {len(df_detail)} 条')
else:
    print(f'警告: 未找到迭代日志文件 {log_file}')
    print('请先运行 纯债基金久期测算.ipynb 生成迭代日志')

找到迭代日志文件: ./output/久期迭代日志_20251231.xlsx
全量基金: 1563 只
触发边界: 1044 只
迭代记录: 3465 条


## 5. 单只基金分析示例

In [9]:
# 选择一只基金进行分析
target_fund = SELECTED_FUNDS[0] if SELECTED_FUNDS else '000033.OF'

print(f'正在分析基金: {target_fund}')

# 获取基金池
fund_pool = calculator.data_preprocessor.get_fund_pool(TARGET_DATE)

# 查找基金信息
fund_info = None
for fund_type in ['short', 'medium_long']:
    df = fund_pool.get(fund_type, pd.DataFrame())
    match = df[df['Code'] == target_fund]
    if not match.empty:
        fund_info = match.iloc[0]
        break

if fund_info is None:
    print(f'未找到基金 {target_fund}')
else:
    print(f'基金名称: {fund_info["Name"]}')

    # 判断基金类型
    fund_bond_type = calculator.fund_classifier.get_fund_type(target_fund, TARGET_DATE)
    print(f'债券类型: {fund_bond_type}')

    # 确定指数列表
    if fund_bond_type == 'rate':
        index_codes = (calculator.index_processor.medium_long_rate_indices 
                      if fund_info.get('fund_type') == 'medium_long' 
                      else calculator.index_processor.short_rate_indices)
    elif fund_bond_type == 'credit':
        index_codes = (calculator.index_processor.medium_long_credit_indices
                      if fund_info.get('fund_type') == 'medium_long'
                      else calculator.index_processor.short_credit_indices)
    else:
        print('未知债券类型')
        index_codes = []

    print(f'使用指数: {len(index_codes)} 个')

    # 获取净值数据
    start_date = (pd.to_datetime(TARGET_DATE) - pd.Timedelta(days=90)).strftime('%Y-%m-%d')
    fund_nav_df = calculator.wind_fetcher.get_fund_nav_smoothed(
        target_fund, start_date, TARGET_DATE
    )

    if fund_nav_df is None:
        print('净值数据获取失败')
    else:
        print(f'净值数据: {len(fund_nav_df)} 条')

        # 获取Wind披露久期
        reported_duration = calculator.wind_fetcher.get_fund_reported_duration(
            target_fund, TARGET_DATE
        )
        print(f'Wind披露久期: {reported_duration:.3f} 年' if reported_duration else 'Wind披露久期: 未知')

        # 获取回归数据
        reg_data = visualizer.get_regression_data(
            target_fund, fund_nav_df, index_codes, TARGET_DATE,
            reported_duration=reported_duration,
            weight_method=WEIGHT_METHOD
        )

        if reg_data is None:
            print('回归数据获取失败（可能没有迭代日志）')
        else:
            print(f'测算久期: {reg_data["calculated_duration"]:.3f} 年')
            print(f'R²: {reg_data["r_squared"]:.4f}')
            print(f'最终因子数: {len(reg_data["final_factors"])} 个')

            # 绘制两页图表
            visualizer.plot_all(
                target_fund, fund_info['Name'], reg_data,
                reported_duration=reported_duration,
                output_dir=output_dir,
                show=SHOW_PLOTS,
                save=True
            )

正在分析基金: 000084.OF
基金名称: 博时安盈A
债券类型: credit
使用指数: 5 个
获取基金 000084.OF 数据失败: ErrorCode=-40521008
净值数据获取失败


In [10]:
# 批量分析多只基金
print(f'\n开始批量分析 {len(SELECTED_FUNDS)} 只基金...')
print('='*60)

success_count = 0
for i, fund_code in enumerate(SELECTED_FUNDS):
    print(f'\n[{i+1}/{len(SELECTED_FUNDS)}] {fund_code}')

    try:
        # 获取基金信息
        fund_info = None
        for fund_type in ['short', 'medium_long']:
            df = fund_pool.get(fund_type, pd.DataFrame())
            match = df[df['Code'] == fund_code]
            if not match.empty:
                fund_info = match.iloc[0]
                break

        if fund_info is None:
            print(f'  跳过: 未找到基金信息')
            continue

        # 判断基金类型
        fund_bond_type = calculator.fund_classifier.get_fund_type(fund_code, TARGET_DATE)

        # 确定指数列表
        if fund_bond_type == 'rate':
            index_codes = (calculator.index_processor.medium_long_rate_indices
                          if fund_info.get('fund_type') == 'medium_long'
                          else calculator.index_processor.short_rate_indices)
        elif fund_bond_type == 'credit':
            index_codes = (calculator.index_processor.medium_long_credit_indices
                          if fund_info.get('fund_type') == 'medium_long'
                          else calculator.index_processor.short_credit_indices)
        else:
            print(f'  跳过: 未知债券类型')
            continue

        # 获取净值数据
        start_date = (pd.to_datetime(TARGET_DATE) - pd.Timedelta(days=90)).strftime('%Y-%m-%d')
        fund_nav_df = calculator.wind_fetcher.get_fund_nav_smoothed(
            fund_code, start_date, TARGET_DATE
        )

        if fund_nav_df is None:
            print(f'  跳过: 净值数据获取失败')
            continue

        # 获取Wind披露久期
        reported_duration = calculator.wind_fetcher.get_fund_reported_duration(
            fund_code, TARGET_DATE
        )

        # 获取回归数据
        reg_data = visualizer.get_regression_data(
            fund_code, fund_nav_df, index_codes, TARGET_DATE,
            reported_duration=reported_duration,
            weight_method=WEIGHT_METHOD
        )

        if reg_data is None:
            print(f'  跳过: 回归数据获取失败')
            continue

        # 绘制图表（不显示，只保存）
        visualizer.plot_all(
            fund_code, fund_info['Name'], reg_data,
            reported_duration=reported_duration,
            output_dir=output_dir,
            show=False,
            save=True
        )

        calc_dur = reg_data['calculated_duration']
        bias = calc_dur - reported_duration if reported_duration else 0
        print(f'  完成: 测算={calc_dur:.3f}Y, 披露={reported_duration:.3f}Y, 偏差={bias:.3f}Y')
        success_count += 1

    except Exception as e:
        print(f'  错误: {str(e)[:50]}')
        continue

print('\n' + '='*60)
print(f'批量分析完成: 成功 {success_count}/{len(SELECTED_FUNDS)} 只基金')
print(f'图表已保存至: {output_dir}')


开始批量分析 1563 只基金...

[1/1563] 000084.OF

[绘图] 000084.OF - 博时安盈A
  [保存] 主面板图: ./output/wls_visualization\WLS_main_000084_OF.png
  [保存] 因子散点图: ./output/wls_visualization\WLS_factors_000084_OF.png
  完成: 测算=1.434Y, 披露=0.343Y, 偏差=1.091Y

[2/1563] 000089.OF

[绘图] 000089.OF - 民生加银高等级信用债C
  [保存] 主面板图: ./output/wls_visualization\WLS_main_000089_OF.png
  [保存] 因子散点图: ./output/wls_visualization\WLS_factors_000089_OF.png
  完成: 测算=0.881Y, 披露=0.641Y, 偏差=0.240Y

[3/1563] 000128.OF
  跳过: 回归数据获取失败

[4/1563] 000322.OF
  跳过: 回归数据获取失败

[5/1563] 000394.OF

[绘图] 000394.OF - 融通通源短融A
  [保存] 主面板图: ./output/wls_visualization\WLS_main_000394_OF.png
  [保存] 因子散点图: ./output/wls_visualization\WLS_factors_000394_OF.png
  完成: 测算=0.218Y, 披露=0.017Y, 偏差=0.201Y

[6/1563] 000503.OF

[绘图] 000503.OF - 中信建投景和中短债A
  [保存] 主面板图: ./output/wls_visualization\WLS_main_000503_OF.png
  [保存] 因子散点图: ./output/wls_visualization\WLS_factors_000503_OF.png
  完成: 测算=0.218Y, 披露=0.126Y, 偏差=0.093Y

[7/1563] 000674.OF

[绘图] 000674.OF - 中海中短债A
  [保

## 7. 分析结果汇总

In [11]:
# 汇总分析结果
summary_results = []

for fund_code in SELECTED_FUNDS:
    log = calculator.duration_model.iteration_logs.get(fund_code)
    if log:
        summary_results.append({
            'fund_code': fund_code,
            'final_duration': log.get('final_duration'),
            'triggered': log.get('triggered'),
            'convergence': log.get('convergence'),
            'total_iterations': log.get('total_iterations'),
            'n_factors': len(log.get('final_factors') or [])
        })

if summary_results:
    summary_df = pd.DataFrame(summary_results)
    print('\n分析结果汇总:')
    print(summary_df.to_string(index=False))

    # 保存汇总结果
    summary_file = os.path.join(output_dir, f'分析汇总_{TARGET_DATE.replace("-", "")}.xlsx')
    summary_df.to_excel(summary_file, index=False)
    print(f'\n汇总结果已保存至: {summary_file}')